# 온라인 쇼핑 세션 데이터 전처리 - KNN / Tree 파이프라인

이 노트북은 `DataPreProcessing.md`에 기술된 2단 전처리 구조를 그대로 실행한다.

1. **공통 전처리** — 데이터 로드 → 결측치 확인 → 타깃 확인 → train/test split
2. **KNN 파이프라인** — 컬럼 축소 → PageValues 누수 판단 → log1p → Scaler → SMOTE
3. **Tree(DT/RF) 파이프라인** — PageValues 처리 → 파생 변수 → One-Hot → ID 유지

모델 학습은 포함하지 않는다. 최종 산출물은 다음 4쌍이다.

- `X_train_knn`, `X_test_knn`, `y_train_knn`, `y_test_knn`
- `X_train_tree`, `X_test_tree`, `y_train_tree`, `y_test_tree`

---

**의존성**: `numpy`, `pandas`, `scikit-learn`, `imbalanced-learn`. `requirements.txt`로 설치된다.

In [ ]:
# 수치/데이터프레임 기본
import numpy as np
import pandas as pd

# 분할은 stratify로 양성 비율 유지, 스케일러는 fit on train만
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# SMOTE: KNN용 학습셋의 양성 부족을 합성 샘플로 보강 (train에만 적용)
from imblearn.over_sampling import SMOTE

# 모든 무작위 단계의 재현성 확보
RANDOM_STATE = 42
pd.set_option('display.max_columns', 40)

## 1. 공통 전처리

세 모델이 같은 train/test 쌍을 쓰도록, split까지는 한 번만 수행한다. fit이 필요한 변환은 split 이후에 두어 누수를 방지한다.

### 1-1. 데이터 로드

In [ ]:
# 루트의 원본 CSV 로드. 행 12,330 / 열 18(타깃 포함) 확인.
df = pd.read_csv('online_shoppers_intention.csv')
print(f'데이터 크기: {df.shape}')
df.head()

In [ ]:
# dtype 확인: 수치형/Boolean/문자열(Month, VisitorType) 구분 파악
df.dtypes

### 1-2. 결측치 확인

UCI 정제본이라 결측 없음을 확인.

In [ ]:
# UCI 정제본이라 결측 0개여야 정상. imputation 단계가 불필요함을 검증.
missing = df.isnull().sum()
print(f'전체 결측 셀: {missing.sum()}')
missing[missing > 0] if missing.sum() > 0 else '결측 없음'

### 1-3. 타깃 변수 확인

`Revenue` 양성 비율이 약 15.5%로 불균형하다. stratify와 클래스 가중치/오버샘플링이 필요한 근거.

In [ ]:
# 타깃을 int로 캐스팅 (Boolean→0/1). 단조 변환이라 누수 위험 없음 → 공통 단계에 둠.
y_full = df['Revenue'].astype(int)

# 양성 비율 약 15.5%인 불균형 확인. stratify와 SMOTE/class_weight의 근거.
print('Revenue 분포 (count):')
print(y_full.value_counts())
print('\nRevenue 분포 (ratio):')
print(y_full.value_counts(normalize=True).round(4))

### 1-4. train/test split (stratify)

모든 모델이 동일한 분할을 공유. 양성 비율을 학습/평가 양쪽에서 유지하기 위해 `stratify=y`.

In [ ]:
# 타깃을 분리한 X. 인코딩/스케일은 split 이후에 (테스트 통계 누수 방지).
X_full = df.drop(columns=['Revenue'])

# 80:20 분할. stratify=y로 학습/평가 양쪽에서 양성 비율(~15.5%) 유지.
# 세 모델이 모두 같은 분할을 공유하도록 random_state 고정.
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_full,
)
print(f'X_train: {X_train.shape},  y_train 양성 비율: {y_train.mean():.4f}')
print(f'X_test : {X_test.shape},  y_test  양성 비율: {y_test.mean():.4f}')

## 2. KNN 전처리 파이프라인

차원 최소화 + 거리 의미 보존이 핵심.

1. `Month`, ID 변수 4종, `VisitorType` 제거
2. `VisitorType` → `is_new_visitor` Boolean 한 비트로 압축
3. `PageValues` 누수 점검 → 제거
4. `Weekend` int 변환 → log1p → StandardScaler
5. SMOTE 오버샘플링 (train만)

### 2-1. 컬럼 선택

In [ ]:
# 공통 분할 결과를 복사해 KNN 전용으로 가공.
X_tr_knn = X_train.copy()
X_te_knn = X_test.copy()

# VisitorType은 One-Hot으로 +3차원 만들지 않고
# 가장 강한 신호인 "신규 방문자 여부" 한 비트로 압축 (구매율 New 24.9% vs Returning 13.9%).
X_tr_knn['is_new_visitor'] = (X_tr_knn['VisitorType'] == 'New_Visitor').astype(int)
X_te_knn['is_new_visitor'] = (X_te_knn['VisitorType'] == 'New_Visitor').astype(int)

# 제거 대상:
#  - Month: 거리 의미가 약함 (1월-12월 거리=11이 부자연스러움), 신호는 다른 행동 지표에 간접 반영됨
#  - OperatingSystems/Browser/Region/TrafficType: ID 정수에 거리 주면 잡음
#  - VisitorType: is_new_visitor로 환원했으므로 원본은 제거
drop_cols = [
    'Month',
    'OperatingSystems', 'Browser', 'Region', 'TrafficType',
    'VisitorType',
]
X_tr_knn = X_tr_knn.drop(columns=drop_cols)
X_te_knn = X_te_knn.drop(columns=drop_cols)

print('KNN용 컬럼:')
print(X_tr_knn.columns.tolist())

### 2-2. PageValues 누수 점검 + 제거

GA에서 `PageValues`는 거래가 발생한 후 역계산되는 값으로, 예측 시점에 알기 어렵다.
분포 검증으로 의심 정황을 확인한 뒤 제거한다.

In [ ]:
# PageValues 누수 점검: 0/>0 그룹의 구매율 격차를 직접 확인
# 격차가 비정상적으로 크면 "거래 이후 역계산된 값"일 가능성이 높음 (예측 시점에 알 수 없는 정보)
check = pd.DataFrame({
    'count':         [(X_tr_knn['PageValues'] == 0).sum(),
                      (X_tr_knn['PageValues'] >  0).sum()],
    'revenue_ratio': [y_train[X_tr_knn['PageValues'] == 0].mean(),
                      y_train[X_tr_knn['PageValues'] >  0].mean()],
}, index=['PageValues == 0', 'PageValues > 0'])
print('누수 의심 점검: PageValues 구간별 구매율')
print(check.round(4))
print('\n→ PageValues > 0 그룹의 구매율이 비정상적으로 높음 (누수 신호). 제거한다.')

# 메인 모델에서는 제거. 비교용으로 살리려면 이 두 줄을 주석 처리.
X_tr_knn = X_tr_knn.drop(columns=['PageValues'])
X_te_knn = X_te_knn.drop(columns=['PageValues'])

### 2-3. 타입 정리 + log1p + StandardScaler

- `Weekend` Boolean → int
- 우편향 수치형에 `log1p` (큰 값의 거리 독점 방지)
- `StandardScaler`로 단위 균등화 (fit은 train에서만)

In [ ]:
# Boolean → int (StandardScaler가 bool을 받지 못함)
X_tr_knn['Weekend'] = X_tr_knn['Weekend'].astype(int)
X_te_knn['Weekend'] = X_te_knn['Weekend'].astype(int)

# 우편향 수치형에 log1p 적용
# 체류 시간/페이지 카운트의 long-tail이 거리 계산을 지배하지 않도록 큰 값을 압축
# log1p(x) = log(1+x): x=0도 처리 가능
skewed = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'SpecialDay',
]
X_tr_knn[skewed] = np.log1p(X_tr_knn[skewed])
X_te_knn[skewed] = np.log1p(X_te_knn[skewed])

# StandardScaler: 평균 0, 분산 1로 통일해 변수 간 거리 기여를 균등화
# fit은 train에서만 → test 통계가 학습으로 누수되지 않음
scaler = StandardScaler().fit(X_tr_knn)
X_tr_knn_s = pd.DataFrame(
    scaler.transform(X_tr_knn),
    columns=X_tr_knn.columns, index=X_tr_knn.index,
)
X_te_knn_s = pd.DataFrame(
    scaler.transform(X_te_knn),
    columns=X_te_knn.columns, index=X_te_knn.index,
)
print(f'KNN 입력 shape: train {X_tr_knn_s.shape}, test {X_te_knn_s.shape}')
X_tr_knn_s.describe().round(3)

### 2-4. SMOTE 오버샘플링 (train만)

거리 기반 합성으로 양성 영역을 점진적으로 채운다. test에는 적용하지 않는다.

In [ ]:
# SMOTE: 양성 샘플 k-NN 이웃 사이의 선분에 합성 점 생성 → 양성 클러스터를 점진적으로 채움
# 거리 기반 합성이라 KNN과 잘 맞음. 스케일링 이후에 적용 (큰 단위 변수가 합성 위치를 지배하지 않도록)
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_tr_knn_res, y_tr_knn_res = sm.fit_resample(X_tr_knn_s, y_train)

# test에는 적용하지 않음: 실제 운영 분포(15.5%)에서 평가해야 정직한 지표가 됨
print(f'SMOTE 전 : {X_tr_knn_s.shape}, 양성 비율 {y_train.mean():.4f}')
print(f'SMOTE 후 : {X_tr_knn_res.shape}, 양성 비율 {y_tr_knn_res.mean():.4f}')

### 2-5. KNN 최종 산출물

In [ ]:
# 최종 KNN 입력. 다른 노트북에서는 이 4개 변수만 import해서 모델 학습에 바로 사용.
X_train_knn = X_tr_knn_res
y_train_knn = y_tr_knn_res
X_test_knn  = X_te_knn_s
y_test_knn  = y_test

print('KNN 최종 산출물')
print(f'  X_train_knn : {X_train_knn.shape}')
print(f'  X_test_knn  : {X_test_knn.shape}')
print(f'  y_train_knn : {y_train_knn.shape}, 양성 비율 {y_train_knn.mean():.4f}')
print(f'  y_test_knn  : {y_test_knn.shape}, 양성 비율 {y_test_knn.mean():.4f}')
print(f'\n입력 컬럼 ({len(X_train_knn.columns)}개):')
print(list(X_train_knn.columns))

## 3. Tree (DT / RF) 전처리 파이프라인

정보 최대 보존 + 명시적 비율 + 사후 가지치기 친화 전략.

1. `PageValues` 제거 (KNN과 동일한 누수 결정)
2. 파생 변수 4개 추가
3. `Month`, `VisitorType` One-Hot
4. ID 변수 유지 + `Weekend` int
5. 불균형은 모델 단계에서 `class_weight='balanced'`로 처리 (여기서는 데이터만 준비)

### 3-1. PageValues 제거 (KNN과 일관)

In [ ]:
# PageValues는 KNN과 같은 누수 판단 결과로 제거
# 같은 데이터 정의 위에서 비교해야 모델 간 성능 차이가 공정하게 해석됨
X_tr_tree = X_train.drop(columns=['PageValues']).copy()
X_te_tree = X_test.drop(columns=['PageValues']).copy()
print(f'PageValues 제거 후 shape: train {X_tr_tree.shape}, test {X_te_tree.shape}')

### 3-2. 파생 변수 생성

트리가 직접 표현하기 어려운 비율/합 신호를 명시적으로 제공해 분할 깊이를 절약한다.

In [ ]:
# 트리는 변수 간 비율/곱을 직접 표현 못 함 → 명시적으로 만들어 분할 깊이 절약
def add_derived(X):
    X = X.copy()
    # 세션 전체 탐색 강도 (페이지 총합)
    X['total_pages'] = (
        X['Administrative'] + X['Informational'] + X['ProductRelated']
    )
    # 세션 전체 체류 시간 합
    X['total_duration'] = (
        X['Administrative_Duration']
        + X['Informational_Duration']
        + X['ProductRelated_Duration']
    )
    # 상품 한 페이지당 평균 체류 (분모에 +1로 0 페이지 방어)
    X['avg_time_per_product'] = (
        X['ProductRelated_Duration'] / (X['ProductRelated'] + 1)
    )
    # 즉시 이탈 vs 점진 이탈 균형 (분모에 epsilon으로 0 방어)
    X['bounce_exit_ratio'] = (
        X['BounceRates'] / (X['ExitRates'] + 1e-6)
    )
    return X

X_tr_tree = add_derived(X_tr_tree)
X_te_tree = add_derived(X_te_tree)

print('파생 변수 추가 완료')
X_tr_tree[['total_pages', 'total_duration',
           'avg_time_per_product', 'bounce_exit_ratio']].describe().round(3)

### 3-3. One-Hot 인코딩 (`Month`, `VisitorType`)

`drop_first=False`로 모든 범주를 살린다. 트리는 다중공선성에 강하므로 손해 없다.
테스트셋은 학습셋 컬럼 기준으로 정렬해야 한다 (test에만 등장하는 카테고리 방어).

In [ ]:
# Month / VisitorType One-Hot
# drop_first=False: 모든 범주를 살림. 트리는 다중공선성에 강하므로 손해 없음
cat_cols = ['Month', 'VisitorType']
X_tr_tree = pd.get_dummies(X_tr_tree, columns=cat_cols, drop_first=False)
X_te_tree = pd.get_dummies(X_te_tree, columns=cat_cols, drop_first=False)

# 테스트셋에만 있거나 빠진 범주가 있을 수 있으므로 train 컬럼 기준으로 정렬
X_te_tree = X_te_tree.reindex(columns=X_tr_tree.columns, fill_value=0)

# get_dummies는 bool로 반환. sklearn 호환을 위해 0/1 정수로 통일
for d in (X_tr_tree, X_te_tree):
    bool_cols = d.select_dtypes(include='bool').columns
    d[bool_cols] = d[bool_cols].astype(int)

print(f'One-Hot 후 shape: train {X_tr_tree.shape}, test {X_te_tree.shape}')

### 3-4. Boolean 변환 (Weekend)

ID 변수(`OperatingSystems`, `Browser`, `Region`, `TrafficType`)는 그대로 유지한다.
트리가 정보 이득으로 사용 여부를 자동 판단한다.

In [ ]:
# Boolean → int (sklearn 호환)
X_tr_tree['Weekend'] = X_tr_tree['Weekend'].astype(int)
X_te_tree['Weekend'] = X_te_tree['Weekend'].astype(int)

# ID 변수는 유지 — 트리는 정보 이득으로 사용 여부를 자동 판단 (KNN과 반대)
print('유지된 ID 변수:')
for c in ['OperatingSystems', 'Browser', 'Region', 'TrafficType']:
    if c in X_tr_tree.columns:
        print(f'  - {c}: 고유값 {X_tr_tree[c].nunique()}개')

### 3-5. Tree 최종 산출물

In [ ]:
# 최종 Tree 입력. y는 SMOTE 없이 원본 분포 유지 (class_weight='balanced'로 모델 단계에서 처리).
X_train_tree = X_tr_tree
X_test_tree  = X_te_tree
y_train_tree = y_train
y_test_tree  = y_test

print('Tree 최종 산출물')
print(f'  X_train_tree : {X_train_tree.shape}')
print(f'  X_test_tree  : {X_test_tree.shape}')
print(f'  y_train_tree : {y_train_tree.shape}, 양성 비율 {y_train_tree.mean():.4f}')
print(f'  y_test_tree  : {y_test_tree.shape}, 양성 비율 {y_test_tree.mean():.4f}')
print(f'\n입력 컬럼 ({len(X_train_tree.columns)}개):')
print(list(X_train_tree.columns))

## 4. (선택) 산출물 저장

다른 노트북에서 바로 이어서 모델 학습을 하고 싶다면 parquet/csv로 저장한다.
기본은 주석 처리해두었으니 필요할 때 풀어 쓴다.

In [ ]:
# 다른 노트북/스크립트에서 바로 모델 학습으로 이어가려면 아래를 활성화
# CSV 외에도 parquet 등 원하는 포맷으로 저장 가능
# import os
# os.makedirs('preprocessed', exist_ok=True)
#
# X_train_knn.to_csv('preprocessed/X_train_knn.csv', index=False)
# X_test_knn .to_csv('preprocessed/X_test_knn.csv',  index=False)
# y_train_knn.to_csv('preprocessed/y_train_knn.csv', index=False)
# y_test_knn .to_csv('preprocessed/y_test_knn.csv',  index=False)
#
# X_train_tree.to_csv('preprocessed/X_train_tree.csv', index=False)
# X_test_tree .to_csv('preprocessed/X_test_tree.csv',  index=False)
# y_train_tree.to_csv('preprocessed/y_train_tree.csv', index=False)
# y_test_tree .to_csv('preprocessed/y_test_tree.csv',  index=False)
# print('preprocessed/ 디렉토리에 저장 완료')

## 5. 요약

| 산출물 | 입력 컬럼 수 | 양성 비율 | 비고 |
|---|---:|---:|---|
| `X_train_knn` | 11 | ~50% (SMOTE 적용 시) | log1p + Scaler + SMOTE |
| `X_test_knn`  | 11 | ~15.5% | log1p + Scaler |
| `X_train_tree` | ~29 | ~15.5% | One-Hot + 파생변수 4개 |
| `X_test_tree`  | ~29 | ~15.5% | 동일 |

두 파이프라인 모두 `PageValues`를 제거했다. 누수 확인용 비교 모델이 필요하면 §2-2 / §3-1에서 `drop` 라인을 주석 처리하고 재실행하면 된다.